# Melia Serengeti - train the site detector on a free GPU (Google Colab)

The NUC / office PC has no GPU (i5-7200U): training there takes many hours.  Colab's free T4 GPU does it in ~20-40 min.

**Before you start (on the PC):**
1. Collect + review images (see `training/TRAINING_GUIDE.md`, steps 1-3).
2. `python training/build_dataset.py dataset_raw/images logs/review --out dataset --tile`
3. Right-click the `dataset` folder -> *Send to* -> *Compressed (zipped) folder* -> `dataset.zip`.
4. Upload `dataset.zip` to your Google Drive, folder `MyDrive/melia/`.

**Here:** *Runtime -> Change runtime type -> T4 GPU*, then run the cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/melia'
!nvidia-smi -L

In [ ]:
!pip -q install ultralytics
!rm -rf /content/dataset && unzip -q $DRIVE/dataset.zip -d /content/
import os, glob
root = '/content/dataset' if os.path.exists('/content/dataset/data.yaml') else os.path.dirname(glob.glob('/content/**/data.yaml', recursive=True)[0])
# data.yaml was written with the PC's Windows path: point it at the Colab copy
txt = open(f'{root}/data.yaml').read().splitlines()
txt = [f'path: {root}' if l.startswith('path:') else l for l in txt]
open(f'{root}/data.yaml', 'w').write('
'.join(txt) + '
')
print(open(f'{root}/data.yaml').read())
for split in ('train', 'val'):
    print(split, len(glob.glob(f'{root}/images/{split}/*')), 'images')

## Train
`yolo11s` on 640 px native-resolution tiles (small birds keep their pixels). With < 300 labelled frames use `epochs=80`; with 1500+ try `model='yolo11m.pt', imgsz=640, epochs=150`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data=f'{root}/data.yaml', imgsz=640, epochs=120, batch=32, device=0, patience=30,
            freeze=10, cos_lr=True, scale=0.6, mosaic=1.0, mixup=0.1, close_mosaic=15,
            hsv_h=0.02, hsv_s=0.6, hsv_v=0.45, degrees=0.0, fliplr=0.5, translate=0.12,
            project='/content/runs', name='melia', exist_ok=True)

## Validate the way the camera runs it: FULL 1920 px frames
Look at recall for **bird** separately - that is the number that matters.

In [ ]:
best = '/content/runs/melia/weights/best.pt'
m = YOLO(best).val(data=f'{root}/data.yaml', imgsz=1920, batch=4, device=0, conf=0.15, plots=True)
print(m.results_dict)
for i, name in m.names.items():
    p, r, ap50, ap = m.box.class_result(i)
    print(f'{name:8s} precision={p:.2f} recall={r:.2f} mAP50={ap50:.2f}')

In [ ]:
import shutil, time
out = f'{DRIVE}/melia_wildlife_{time.strftime("%Y%m%d")}.pt'
shutil.copy(best, out)
print('Saved to Drive:', out)
print('On the PC: copy it to models/melia_wildlife.pt and set in .env:')
print('  WILDLIFE_CUSTOM_YOLO_ENABLED=true')
print('  WILDLIFE_CUSTOM_YOLO_MODEL_PATH=models/melia_wildlife.pt')